# Subnetwork Analysis

Identifies minimal circuits, evaluates faithfulness and minimality, compares subnetworks, and performs greedy circuit search.

**References:**
- Conmy et al. (2023) "Towards Automated Circuit Discovery for Mechanistic Interpretability"
- Wang et al. (2023) "Interpretability in the Wild"

## Why This Matters

Subnetwork analysis extracts minimal circuits that faithfully reproduce model behavior on specific tasks. Faithfulness, minimality, and comparison metrics let you evaluate whether a discovered subnetwork truly captures the relevant computation or misses important components.

**Key references:**
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.subnetwork_analysis import (
    extract_important_components,
    subnetwork_faithfulness,
    subnetwork_minimality,
    compare_subnetworks,
    greedy_subnetwork_search,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
def metric(logits): return float(logits[-1, 0])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Extract important components via ablation
imp = extract_important_components(model, tokens, metric, threshold=0.1)
print(f"Important heads ({len(imp['important_heads'])}): {imp['important_heads']}")
print(f"Important MLPs ({len(imp['important_mlps'])}): {imp['important_mlps']}")
print(f"Fraction important: {imp['fraction_important']:.2%}")
print(f"\nTop ablation effects:")
sorted_fx = sorted(imp['ablation_effects'].items(), key=lambda x: x[1], reverse=True)
for comp, eff in sorted_fx[:8]:
    print(f"  {comp}: effect={eff:.6f}")

In [ ]:
# 2. Subnetwork faithfulness
heads = imp['important_heads']
mlps = imp['important_mlps']
faith = subnetwork_faithfulness(model, tokens, metric, heads, mlps)
print(f"Full metric: {faith['full_metric']:.6f}")
print(f"Subnetwork metric: {faith['subnetwork_metric']:.6f}")
print(f"Faithfulness: {faith['faithfulness']:.4f}")
print(f"Relative error: {faith['relative_error']:.4f}")

In [ ]:
# 3. Subnetwork minimality
mini = subnetwork_minimality(model, tokens, metric, heads, mlps, threshold=0.5)
print(f"Is minimal: {mini['is_minimal']}")
print(f"Removable heads: {mini['removable_heads']}")
print(f"Removable MLPs: {mini['removable_mlps']}")
print(f"Component necessity:")
for comp, necessary in mini['component_necessity'].items():
    print(f"  {comp}: {'necessary' if necessary else 'removable'}")

In [ ]:
# 4. Compare subnetworks
sub_a = {"heads": [(0, 0), (0, 1), (1, 0)], "mlps": [0, 1]}
sub_b = {"heads": [(1, 0), (1, 1), (2, 0)], "mlps": [1, 2]}
comp = compare_subnetworks(model, tokens, metric, sub_a, sub_b)
print(f"Faithfulness A: {comp['faithfulness_a']:.4f} (size={comp['size_a']})")
print(f"Faithfulness B: {comp['faithfulness_b']:.4f} (size={comp['size_b']})")
print(f"Jaccard similarity: {comp['jaccard_similarity']:.4f}")
print(f"Overlap heads: {comp['overlap_heads']}")
print(f"Overlap MLPs: {comp['overlap_mlps']}")

In [ ]:
# 5. Greedy subnetwork search
search = greedy_subnetwork_search(model, tokens, metric, target_faithfulness=0.8)
print(f"Components needed: {search['n_components_needed']}")
print(f"Final faithfulness: {search['final_faithfulness']:.4f}")
print(f"Selected heads: {search['selected_heads']}")
print(f"Selected MLPs: {search['selected_mlps']}")
print(f"\nFaithfulness trajectory:")
for i, f in enumerate(search['faithfulness_trajectory']):
    print(f"  +{i+1} components: faithfulness={f:.4f}")